# Clinic Case Study — Phase 3: Full Three-Stage Clinic

**Case study**: Community Health Clinic | **Phase**: 3 of 5

## Learning Objectives
By the end of this notebook you will be able to:
1. Instantiate and run the full `ClinicModel` from the `simdes` package.
2. Interpret all five output metrics: `mean_total_time`, `mean_wait_registration`, `mean_wait_triage`, `mean_wait_exam`, `n_patients`.
3. Visualise per-stage waiting time distributions across replications.
4. Identify which stage causes the longest patient delay in the baseline configuration.

---
> Phase 3 completes the patient flow: Registration → Triage → Examination room.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from simdes.models.clinic import ClinicModel, ClinicParams
from simdes.analysis import confidence_interval

## Full System Description

```
Arrival
  │
  ▼
[Queue] ─► 1 Registration clerk (Exp 3 min)
  │
  ▼
[Queue] ─► 1 Triage nurse (Exp 8 min)
  │
  ▼
[Queue] ─► 2 Exam rooms (Exp 15 min)
  │
  ▼
Exit
```

| Stage | Servers | Mean svc | Utilisation |
|---|---|---|---|
| Registration | 1 | 3 min | 0.25 |
| Triage nurse | 1 | 8 min | 0.667 |
| Exam rooms | 2 | 15 min | 0.625 |

In [ ]:
baseline = ClinicParams(
    n_registration=1,
    n_nurses=1,
    n_exam_rooms=2,
    arrival_rate=5.0 / 60.0,   # 5 patients/hour
    reg_mean=3.0,
    triage_mean=8.0,
    exam_mean=15.0,
    sim_time=480.0,
)

model = ClinicModel(params=baseline, seed=42)
df = model.run_replications(30)
df.head()

In [ ]:
# Overall summary
wait_cols = ['mean_wait_registration', 'mean_wait_triage', 'mean_wait_exam', 'mean_total_time']
for col in wait_cols:
    m, lo, hi = confidence_interval(df[col].to_numpy())
    print(f'{col:32s}: {m:6.2f} min  95% CI [{lo:.2f}, {hi:.2f}]')

In [ ]:
# Box plots — per-stage waiting times across replications
fig, ax = plt.subplots(figsize=(8, 4))
labels = ['Registration\nwait', 'Triage\nwait', 'Exam\nwait', 'Total\ntime']
data   = [df[c].to_numpy() for c in wait_cols]
ax.boxplot(data, labels=labels, patch_artist=True,
           boxprops=dict(facecolor='tab:cyan', alpha=0.6))
ax.set_ylabel('Mean time per replication (min)')
ax.set_title('Phase 3 — Full clinic: per-stage waits across 30 replications')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Breakdown: what fraction of total time is waiting vs. receiving service?
wait_total  = (df['mean_wait_registration'] + df['mean_wait_triage'] + df['mean_wait_exam']).mean()
svc_total   = 3.0 + 8.0 + 15.0   # sum of expected service times
grand_total = df['mean_total_time'].mean()

print(f'Sum of mean stage waits : {wait_total:.2f} min')
print(f'Sum of expected svc times: {svc_total:.2f} min')
print(f'Mean total time (sim)    : {grand_total:.2f} min')
print(f'% time waiting           : {100*wait_total/grand_total:.1f}%')

In [ ]:
# Stacked bar: average time breakdown per patient
means = {c: df[c].mean() for c in wait_cols[:3]}
svc   = {'reg_svc': 3.0, 'triage_svc': 8.0, 'exam_svc': 15.0}

fig, ax = plt.subplots(figsize=(6, 4))
bottom = 0
colors = ['#1f77b4', '#4a9acb', '#ff7f0e', '#ffb55e', '#2ca02c', '#72cc72']
labels_bar = ['Wait reg', 'Svc reg', 'Wait triage', 'Svc triage', 'Wait exam', 'Svc exam']
values     = [means['mean_wait_registration'], svc['reg_svc'],
              means['mean_wait_triage'],        svc['triage_svc'],
              means['mean_wait_exam'],           svc['exam_svc']]

for v, c, l in zip(values, colors, labels_bar):
    ax.bar('Patient journey', v, bottom=bottom, color=c, label=f'{l}: {v:.1f} min')
    bottom += v

ax.set_ylabel('Minutes')
ax.set_title('Average patient time breakdown')
ax.legend(loc='upper left', fontsize=8, bbox_to_anchor=(1.01, 1))
fig.tight_layout()
plt.show()

## Summary

The full three-stage clinic model reveals that:
- The **triage nurse** dominates the waiting time despite having the second-highest utilisation.
- Patients spend a significant fraction of their visit waiting (not receiving care).
- Simulation handles the tandem structure naturally — no approximation needed.

In Phase 4 we will run **scenarios** to find the staffing configuration that balances
wait time against cost.

## Try It Yourself

1. Change `n_exam_rooms=1`.  How does the exam waiting time change? Is this stage now the bottleneck?
2. What arrival rate would push the triage nurse to ρ = 0.9? Run the simulation at that rate.
3. Run 100 replications instead of 30. Does the CI for `mean_total_time` include 26 minutes (a common target)?